# Silver Branch Implementation

Reads unmerged bronze branches, cleans and validates them, MERGEs valid rows into silver,
quarantines bad rows, and flags processed bronze rows.

| Outcome | Destination | Bronze `merge_flag` |
|---|---|---|
| Valid | `silver.branch` | `true` |
| Hard reject (bad data) | `silver.quarantine_branch` | `true` |

**Job order:** runs first, in parallel with Silver_Customer. Silver_Agent depends on it.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.branch (
  branch_id        INT,
  branch_country   STRING,
  branch_city      STRING,
  merged_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/branch';

CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.quarantine_branch (
  branch_id             INT,
  branch_country        STRING,
  branch_city           STRING,
  reject_reason         STRING,
  quarantined_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/quarantine/branch';

In [0]:
from pyspark.sql import functions as F, Window

COLUMNS = ["branch_id", "branch_country", "branch_city"]

# 1. Read only unmerged bronze branches
branch_df = spark.sql("SELECT * FROM policysystemdatabricks.bronze.branch WHERE merge_flag = false")

# 2. Clean: remove leading/trailing/double spaces; branch_country in UPPER CASE; default missing values
clean_df = branch_df.selectExpr(
    "branch_id",
    r"upper(coalesce(nullif(regexp_replace(trim(branch_country), '\\s+', ' '), ''), 'Unknown')) AS branch_country",
    r"coalesce(nullif(regexp_replace(trim(branch_city), '\\s+', ' '), ''), 'Unknown') AS branch_city",
)

# 3. Deduplicate: one row per branch_id (no timestamp in the source, so the pick is alphabetical but deterministic).
#    NULL branch_id rows are kept individually so each one is quarantined.
w = Window.partitionBy("branch_id").orderBy("branch_country", "branch_city")
df_result = (clean_df.withColumn("rn", F.row_number().over(w))
             .filter("rn = 1 OR branch_id IS NULL")
             .drop("rn"))

In [0]:
# \p{L} = any Unicode letter, so names like Sao Paulo / Zurich with accents are accepted.
NAME_PATTERN = r"^\p{L}[\p{L} .'\-]*$"

reject_reason = (
    F.when(F.col("branch_id").isNull(), "null_branch_id")
     .when(~F.col("branch_country").rlike(NAME_PATTERN), "invalid_branch_country")
     .when(~F.col("branch_city").rlike(NAME_PATTERN), "invalid_branch_city")
)

classified_df = df_result.withColumn("reject_reason", reject_reason)
classified_df.createOrReplaceTempView("branch_classified")

display(classified_df.groupBy(F.coalesce("reject_reason", F.lit("VALID")).alias("outcome")).count())

outcome,count


In [0]:
display(spark.sql("""
MERGE INTO policysystemdatabricks.silver.branch AS t
USING (SELECT * FROM branch_classified WHERE reject_reason IS NULL) AS s
ON t.branch_id = s.branch_id
WHEN MATCHED THEN UPDATE SET
  t.branch_country   = s.branch_country,
  t.branch_city      = s.branch_city,
  t.merged_timestamp = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (branch_id, branch_country, branch_city, merged_timestamp)
  VALUES (s.branch_id, s.branch_country, s.branch_city, current_timestamp())
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
display(spark.sql(f"""
INSERT INTO policysystemdatabricks.silver.quarantine_branch
SELECT {", ".join(COLUMNS)}, reject_reason, current_timestamp()
FROM branch_classified
WHERE reject_reason IS NOT NULL
"""))

num_affected_rows,num_inserted_rows
0,0


In [0]:
display(spark.sql("""
MERGE INTO policysystemdatabricks.bronze.branch AS b
USING (SELECT DISTINCT branch_id FROM branch_classified WHERE branch_id IS NOT NULL) AS s
ON b.branch_id = s.branch_id AND b.merge_flag = false
WHEN MATCHED THEN UPDATE SET b.merge_flag = true
"""))

# NULL keys cannot be matched in a MERGE; they are always quarantined above, so flag them directly.
display(spark.sql("""
UPDATE policysystemdatabricks.bronze.branch
SET merge_flag = true
WHERE merge_flag = false AND branch_id IS NULL
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


num_affected_rows
0


In [0]:
%sql
SELECT (SELECT count(*) FROM policysystemdatabricks.bronze.branch WHERE merge_flag = false) AS bronze_unmerged_rows,  -- should be 0
       (SELECT count(*) FROM policysystemdatabricks.silver.quarantine_branch) AS quarantined_rows_total,
       count(*)                                     AS silver_rows,
       count(DISTINCT branch_id)                    AS silver_distinct_branches,
       count_if(branch_country = 'UNKNOWN')         AS unknown_country,
       count_if(branch_country != upper(trim(branch_country))) AS country_not_upper_trimmed,  -- should be 0
       count_if(branch_city = 'Unknown')            AS unknown_city,
       count_if(merged_timestamp IS NULL)           AS rows_missing_merged_timestamp  -- should be 0
FROM policysystemdatabricks.silver.branch

bronze_unmerged_rows,quarantined_rows_total,silver_rows,silver_distinct_branches,unknown_country,country_not_upper_trimmed,unknown_city,rows_missing_merged_timestamp
0,0,1000,1000,56,0,56,0
